# Evaluation

This notebook contains the code used to perform the evaluation of the resulting tracking system. We compare, using a small set of test sequences, the performance of some different configurations.

In [7]:
import os

import pandas as pd

In [2]:
import util
import visualize
import dataset
import detect
import tracker

In [ ]:
# This code is just here so I can reload the external files in case I make
# changes to them after having already imported them.
import importlib

importlib.reload(util)
importlib.reload(visualize)
importlib.reload(dataset)
importlib.reload(detect)
importlib.reload(tracker)

#### Running the System on the Test Sequences

First, we will run the system on all the test sequences. We will run it using the five available sizes of the YOLO model used by the system and using a different number of available camera views.

In [4]:
data = dataset.CmuPanopticDataset()

In [ ]:
physics = tracker.build_physics()
physics.to("cuda")

for model in ["yolo26n-pose", "yolo26s-pose", "yolo26m-pose", "yolo26l-pose"]:
    detector = detect.PoseDetector(model_name=model)
    detector.to("cuda")
    for scene in data.test_scenes:
        # We only use more than 4 cameras for the smallest model.
        for n_cams in [4, 8, 12, 16] if model == "yolo26n-pose" else [4]:
            print(
                f"running for {scene} with {n_cams} cameras using {model}...")
            # Only perform if we don't have the result yet, to enable resuming.
            if not os.path.exists(f"./stats/eval/{model}/{n_cams}/{scene}-times.json"):
                source = data.get_source(scene, num_vga_cams=n_cams)
                source.to("cuda")
                track = tracker.Tracker(detector, physics)
                cams, frames, fps = track.evaluate(source, progress=0)
                util.save_tracks(
                    f"./stats/eval/{model}/{n_cams}/{scene}.json", cams, frames, fps)
                util.save_json(
                    f"./stats/eval/{model}/{n_cams}/{scene}-times.json", track.timing_stats)

running for 171204_pose6 with 4 cameras using yolo26n-pose...
running for 171204_pose6 with 8 cameras using yolo26n-pose...
running for 171204_pose6 with 12 cameras using yolo26n-pose...
running for 171204_pose6 with 16 cameras using yolo26n-pose...
running for 161029_piano4 with 4 cameras using yolo26n-pose...
running for 161029_piano4 with 8 cameras using yolo26n-pose...
running for 161029_piano4 with 12 cameras using yolo26n-pose...
running for 161029_piano4 with 16 cameras using yolo26n-pose...
running for 170915_office1 with 4 cameras using yolo26n-pose...
running for 170915_office1 with 8 cameras using yolo26n-pose...
running for 170915_office1 with 12 cameras using yolo26n-pose...
running for 170915_office1 with 16 cameras using yolo26n-pose...
running for 161029_build1 with 4 cameras using yolo26n-pose...
running for 161029_build1 with 8 cameras using yolo26n-pose...
running for 161029_build1 with 12 cameras using yolo26n-pose...
running for 161029_build1 with 16 cameras using 

Next to actually do the evaluation we also need the ground truth tracks. I extract them here ones into the same format as the track predictions, and then we can load them again for computing evaluation metrics.

In [ ]:
for scene in data.test_scenes:
    cams_gt, frames_gt, fps_gt = data.load_ground_truth_vga(scene)
    util.save_tracks(f"./stats/gt/{scene}.json", cams_gt, frames_gt, fps_gt)

Perform performing visualizations of the results, we need to actually compute the metrics for each evaluation run.

In [ ]:
for model in ["yolo26n-pose", "yolo26s-pose", "yolo26m-pose", "yolo26l-pose"]:
    for scene in data.test_scenes:
        for n_cams in [4, 8, 12, 16] if model == "yolo26n-pose" else [4]:
            print(
                f"running for {scene} with {n_cams} cameras using {model}...")
            # Only perform if we don't have the result yet, to enable resuming.
            if not os.path.exists(f"./stats/eval/{model}/{n_cams}/{scene}-metrics.json"):
                metrics = util.evaluate_from_files(
                    f"./stats/gt/{scene}.json", f"./stats/eval/{model}/{n_cams}/{scene}.json")
                util.save_json(
                    f"./stats/eval/{model}/{n_cams}/{scene}-metrics.json", metrics)

running for 171204_pose6 with 4 cameras using yolo26n-pose...
running for 171204_pose6 with 8 cameras using yolo26n-pose...
running for 171204_pose6 with 12 cameras using yolo26n-pose...
running for 171204_pose6 with 16 cameras using yolo26n-pose...
running for 161029_piano4 with 4 cameras using yolo26n-pose...
running for 161029_piano4 with 8 cameras using yolo26n-pose...
running for 161029_piano4 with 12 cameras using yolo26n-pose...
running for 161029_piano4 with 16 cameras using yolo26n-pose...
running for 170915_office1 with 4 cameras using yolo26n-pose...
running for 170915_office1 with 8 cameras using yolo26n-pose...
running for 170915_office1 with 12 cameras using yolo26n-pose...
running for 170915_office1 with 16 cameras using yolo26n-pose...
running for 161029_build1 with 4 cameras using yolo26n-pose...
running for 161029_build1 with 8 cameras using yolo26n-pose...
running for 161029_build1 with 12 cameras using yolo26n-pose...
running for 161029_build1 with 16 cameras using 

Now that we have the evaluation results for all sequences, we merge them into a large dataset for further processing.

In [9]:
df = []
for model in ["yolo26n-pose", "yolo26s-pose", "yolo26m-pose", "yolo26l-pose"]:
    for scene in data.test_scenes:
        for n_cams in [4, 8, 12, 16] if model == "yolo26n-pose" else [4]:
            metrics = util.load_json(f"./stats/eval/{model}/{n_cams}/{scene}-metrics.json")
            times = util.load_json(f"./stats/eval/{model}/{n_cams}/{scene}-times.json")
            df.append({
                "model": model, "scene": scene, "n_cams": n_cams,
                "MPJPE": metrics["MPJPE"], "MOTA": metrics["MOTA"],
                "MOTP": metrics["MOTP (Avg Miss Distance)"],
                "Precision": metrics["Precision"], "Recall": metrics["Recall"],
                "t_detection": times["detection"][0] / times["detection"][1],
                "t_matching_old": times["matching_old"][0] / times["matching_old"][1],
                "t_matching_new": times["matching_new"][0] / times["matching_new"][1],
                "t_update": times["update"][0] / times["update"][1],
                "t_create_new": times["create_new"][0] / times["create_new"][1],
                "t_prediction": times["prediction"][0] / times["prediction"][1],
            })
df = pd.DataFrame(df)
df.to_csv("stats/results.csv", index=False)

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   model           35 non-null     str    
 1   scene           35 non-null     str    
 2   n_cams          35 non-null     int64  
 3   MPJPE           35 non-null     float64
 4   MOTA            35 non-null     float64
 5   MOTP            35 non-null     float64
 6   Precision       35 non-null     float64
 7   Recall          35 non-null     float64
 8   t_detection     35 non-null     float64
 9   t_matching_old  35 non-null     float64
 10  t_matching_new  35 non-null     float64
 11  t_update        35 non-null     float64
 12  t_create_new    35 non-null     float64
 13  t_prediction    35 non-null     float64
dtypes: float64(11), int64(1), str(2)
memory usage: 4.0 KB


#### Performance on Test Set

Next we will examine the performance of different configurations of the tracking system on the held-out test sequences. We evaluate using both pose estimation and object tracking related metrics.

In [ ]:
# TODO

#### Qualitative Results

In this section we just look at some example sequence to compare how the differently configured tracking systems perform.

In [ ]:
# TODO